# 1. What you'll learn

        - see core, border, and noise logic
- measure DBSCAN sensitivity to eps
- use HDBSCAN for variable density

        **The one question this notebook answers:** How can density connectivity recover curved clusters and label points as noise?

# 2. Setup

This lesson keeps dbscan and hdbscan small enough to inspect end to end. Fixed seeds and one shared split make every comparison reproducible.

The seed is fixed at 0 for data, splits, optimization, and sampling.
Versions are printed so this execution remains an auditable experiment,
not a result detached from the software that produced it.

In [ ]:
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.datasets import make_moons,make_blobs
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score,silhouette_score
import hdbscan

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

Two noisy moons plus injected uniform noise show arbitrary shape. A second dataset combines one dense and one diffuse blob to stress a single eps.

**Small example:** With min_samples=3, five points on a chain can form one cluster if each core neighbourhood overlaps the next—even when the endpoints are far apart.

In [ ]:
rng=np.random.default_rng(SEED);core,truth_core=make_moons(n_samples=600,noise=.065,random_state=SEED);noise=rng.uniform([-1.5,-.8],[2.5,1.5],size=(90,2));X=np.vstack([core,noise]);truth=np.r_[truth_core,np.repeat(-1,len(noise))];print("Shape:",X.shape,"known moon/noise counts:",pd.Series(truth).value_counts().to_dict())

# 4. Explore

The three views establish the target, the useful structure, and the specific data property that will govern the model's success or failure.

The next code produces three purposeful panels. Read the written
takeaways underneath before moving on; the plots are evidence for later
modeling choices, not decoration.

In [ ]:
d=np.sort(np.partition(np.linalg.norm(X[:,None,:]-X[None,:,:],axis=2),5,axis=1)[:,5]);fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].scatter(X[:,0],X[:,1],c=truth,cmap="viridis",s=12);axes[0].set_title("Curves plus planted noise")
axes[1].plot(d,color="#176b66");axes[1].set(title="Sorted 5-neighbour distance",ylabel="distance")
sizes=[np.sum(truth==v) for v in [-1,0,1]];axes[2].bar(["noise","moon A","moon B"],sizes,color=[".5","#176b66","#d38b45"]);axes[2].set_title("Evaluation-only truth");plt.tight_layout();plt.show()

**Takeaway:** The clusters curve around one another, defeating centroid partitions.

**Takeaway:** The k-distance bend offers evidence for eps but not a universal answer.

**Takeaway:** Noise is a legitimate output class, not a third semantic group.

# 5. Prepare

We create the evaluation boundary before learning any transformation or model choice. Training-only operations remain inside the fitted workflow so held-out rows cannot influence them.

Any learned scaling, encoding, imputation, or resampling belongs after
the split—normally inside a pipeline. Doing it on all rows would let
held-out information influence training and would make evaluation too
optimistic.

In [ ]:
Xs=StandardScaler().fit_transform(X);eps_values=[.10,.15,.20,.28,.40];models={e:DBSCAN(eps=e,min_samples=6).fit(Xs) for e in eps_values};print("Scaled shape:",Xs.shape)

# 6. Train

        Training turns the assumptions behind dbscan and hdbscan into an explicit fitted rule. The printed intermediate values expose what was learned rather than treating fit() as magic.

        Conceptually, fitting follows these steps:

        1. Count neighbours inside eps for every point.
2. Mark points with at least min_samples as core.
3. Connect density-reachable core neighbourhoods.
4. Attach reachable border points and label the rest -1 as noise.

In [ ]:
for e,m in models.items(): print(f"eps={e:.2f}: clusters={len(set(m.labels_)-{-1})}, noise={(m.labels_==-1).mean():.1%}")
hdb=hdbscan.HDBSCAN(min_cluster_size=35,min_samples=6).fit(Xs);model=models[.20];print("HDBSCAN clusters:",len(set(hdb.labels_)-{-1}),"noise:",round(np.mean(hdb.labels_==-1),3))

# 7. Evaluate

We compare against a deliberately simple baseline, report a task-appropriate metric, and save three diagnostic views that explain where the score came from.

We report a baseline as well as the model and save the central figure as
`key_figure.png`. A model earns attention only by beating a simple,
honestly evaluated alternative.

In [ ]:
mask=truth!=-1;ari_db=adjusted_rand_score(truth[mask],model.labels_[mask]);ari_h=adjusted_rand_score(truth[mask],hdb.labels_[mask]);base=0
fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].plot(eps_values,[len(set(m.labels_)-{-1}) for m in models.values()],marker="o",label="clusters");axes[0].plot(eps_values,[np.mean(m.labels_==-1)*3 for m in models.values()],marker="o",label="3× noise share");axes[0].set(title="DBSCAN sensitivity",xlabel="eps");axes[0].legend()
axes[1].scatter(X[:,0],X[:,1],c=model.labels_,cmap="tab10",s=12);axes[1].set_title(f"DBSCAN eps=.20 · ARI {ari_db:.3f}")
axes[2].scatter(X[:,0],X[:,1],c=hdb.labels_,cmap="tab10",s=12);axes[2].set_title(f"HDBSCAN · ARI {ari_h:.3f}");plt.tight_layout();plt.savefig("key_figure.png",dpi=160,bbox_inches="tight");plt.show();print("Chance-adjusted random ARI baseline≈0")

**Takeaway:** The model improves on the simple baseline where its assumptions match the data; the diagnostics show that the aggregate score alone is not enough.

# 8. Break it

We now violate the important condition directly: A single DBSCAN eps cannot simultaneously describe a very dense cluster and a diffuse one. The resulting number makes the cost of that violation visible.

The following experiment deliberately violates an assumption or removes
a protection. Compare the printed damage with the healthy evaluation;
the failure is part of the lesson, not an accidental exception.

In [ ]:
dense,_=make_blobs(n_samples=250,centers=[[-2,0]],cluster_std=.18,random_state=SEED);diffuse,_=make_blobs(n_samples=250,centers=[[2,0]],cluster_std=.85,random_state=SEED);V=StandardScaler().fit_transform(np.vstack([dense,diffuse]));vtruth=np.r_[np.zeros(250),np.ones(250)]
for e in [.12,.25,.45]: print(f"eps={e}: ARI={adjusted_rand_score(vtruth,DBSCAN(eps=e,min_samples=8).fit_predict(V)):.3f}")
print("HDBSCAN variable-density ARI:",round(adjusted_rand_score(vtruth,hdbscan.HDBSCAN(min_cluster_size=30).fit_predict(V)),3))

**Use this when…** clusters are dense connected shapes and explicit noise labels are useful.

        **Don't use this when…** dimensions are high, densities change dramatically without HDBSCAN, or every point must receive a stable segment.

        ## Try this

        1. Vary min_samples and track noise share.
2. Add 20 noise features and observe distance concentration.
3. Inspect HDBSCAN membership probabilities rather than hard labels.